# 🧠 ความสมดุลระหว่างความเอนเอียงและความแปรปรวน (Bias-Variance Tradeoff): การแยกส่วนเชิงตัวเลข

ยินดีต้อนรับสู่สมุดโน้ตอธิบายการใช้งานจริงสำหรับ **Bias-Variance Tradeoff**! ในสมุดโน้ตเล่มนี้ เราจะ:
1. สร้างฟังก์ชันไซน์ที่ไม่เป็นเชิงเส้น (non-linear sinusoidal function) ร่วมกับสัญญาณรบกวน (noise)
2. ฟิตโมเดลสมการถดถอยพหุนาม (polynomial regression) ที่ระดับองศา (degree) ต่างๆ (1, 4, 15) ซึ่งเป็นตัวแทนของความซับซ้อนของโมเดลในระดับที่แตกต่างกัน
3. คำนวณ **การแยกส่วนทางคณิตศาสตร์ของความเอนเอียง (bias) และความแปรปรวน (variance)** จากศูนย์ (from scratch) โดยการฝึกฝนโมเดลบนชุดข้อมูลอิสระหลายชุดที่ดึงมาจากฟังก์ชันพื้นฐานเดียวกัน
4. แสดงภาพให้เห็นว่าการทำนายของโมเดลมีความผันผวน (variance) อย่างไร เปรียบเทียบกับสัดส่วนที่ผลทำนายคลาดเคลื่อนไปจากค่าจริง (bias)
5. พล็อต **เส้นโค้งรูปตัว U ของ Bias-Variance** แบบดั้งเดิม ซึ่งแสดงความสัมพันธ์ระหว่างความซับซ้อนของโมเดลและความคลาดเคลื่อนที่คาดหวังจากการทดสอบ (expected test error)
6. เชื่อมโยงข้อสังเกตเหล่านี้กับการเลือกโมเดลใน YOLO (เช่น YOLO Nano vs. YOLO Extra Large)

เรามาเริ่มด้วยการนำเข้าไลบรารีที่จำเป็นกันเลยครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# Set seed for reproducibility
np.random.seed(42)

## 1. การสร้างฟังก์ชันจริงและชุดข้อมูลจำลอง (Simulating the True Function and Datasets)
ฟังก์ชันจริงของเราคือ $f(x) = \cos(1.5 \pi x)$ เราจะสร้างชุดข้อมูลฝึกฝนจำนวนหลายชุด โดยแต่ละชุดมีขนาด 20 ตัวอย่าง และสุ่มเพิ่มสัญญาณรบกวนแบบ Gaussian (Gaussian noise) $\epsilon \sim \mathcal{N}(0, 0.2^2)$

In [ ]:
def true_func(x):
    return np.cos(1.5 * np.pi * x)

# Test inputs (to evaluate bias and variance)
x_test = np.linspace(0, 1, 100)
y_true = true_func(x_test)

# Generate one sample training set for visualization
x_train = np.sort(np.random.rand(20))
y_train = true_func(x_train) + np.random.normal(0, 0.2, 20)

plt.figure(figsize=(8, 5))
plt.plot(x_test, y_true, color='black', linewidth=2, label='True Function f(x)')
plt.scatter(x_train, y_train, color='red', edgecolor='k', s=40, label='Noisy Training Set')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Sinusoidal Data Generating Process')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 2. การแสดงภาพการเรียนรู้น้อยเกินไป (Underfitting) vs. การเรียนรู้มากเกินไป (Overfitting)

เรามาลองฟิตพหุนามระดับดีกรี 1 (เชิงเส้น), ดีกรี 4 (ระดับที่เหมาะสมที่สุด) และดีกรี 15 (เรียนรู้มากเกินไป) เข้ากับชุดข้อมูลฝึกฝนของเรากันครับ

In [ ]:
degrees = [1, 4, 15]

plt.figure(figsize=(16, 5))
for idx, degree in enumerate(degrees):
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(x_train[:, np.newaxis], y_train)
    
    y_pred = model.predict(x_test[:, np.newaxis])
    
    plt.subplot(1, 3, idx + 1)
    plt.plot(x_test, y_true, color='black', label='True f(x)')
    plt.plot(x_test, y_pred, color='blue', linewidth=2, label=f'Degree {degree}')
    plt.scatter(x_train, y_train, color='red', edgecolor='k', s=35)
    plt.ylim(-2, 2)
    plt.title(f"Polynomial Degree {degree}")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

-   **Degree 1 (Underfitting):** เส้นตรงแบบง่าย มีความเอนเอียงสูง (High bias เนื่องจากไม่สามารถจับความโค้งของฟังก์ชันได้) แต่มีความแปรปรวนต่ำ (low variance)
-   **Degree 4 (Sweet Spot):** จับรูปแบบความโค้งได้อย่างสวยงาม มีความเอนเอียงต่ำ และมีความแปรปรวนต่ำ (ค่าที่เหมาะสมที่สุด)
-   **Degree 15 (Overfitting):** เส้นมีความคดเคี้ยวอย่างรุนแรงเพื่อพยายามลากผ่านทุกๆ จุดข้อมูลที่มีสัญญาณรบกวน มีความเอนเอียงต่ำมากบนข้อมูลฝึกฝน แต่มีความแปรปรวนสูงมาก (high variance ซึ่งส่งผลให้การทำนายข้อมูลทดสอบไม่มีความเสถียรอย่างยิ่งครับ)

## 3. การแยกส่วนหาความเอนเอียงและความแปรปรวนเชิงตัวเลข (Decomposing Bias and Variance Numerically)

เพื่อวัดค่าความเอนเอียง (Bias) และความแปรปรวน (Variance) โดยตรง เราจะสร้างชุดข้อมูลฝึกฝนที่เป็นอิสระต่อกันจำนวน $N = 100$ ชุด ฟิตโมเดลในแต่ละชุดข้อมูล และเก็บผลลัพธ์การทำนายบนชุดข้อมูลทดสอบของเราครับ
จากนั้นเราจะคำนวณ:
-   **ค่าเฉลี่ยของการทำนายจากโมเดล (Average Model Prediction):** $\mathbb{E}[\hat{f}(x)]$
-   **ค่าความเอนเอียงกำลังสอง (Bias$^2$):** $(\mathbb{E}[\hat{f}(x)] - f(x))^2$
-   **ค่าความแปรปรวน (Variance):** $\mathbb{E}[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2]$

In [ ]:
n_datasets = 100
n_samples = 20
degrees_sweep = [1, 2, 4, 8, 15]

# Arrays to store predictions: shape (n_degrees, n_datasets, n_test_points)
all_preds = np.zeros((len(degrees_sweep), n_datasets, len(x_test)))

for d_idx, degree in enumerate(degrees_sweep):
    for i in range(n_datasets):
        x_tr = np.random.rand(n_samples)
        y_tr = true_func(x_tr) + np.random.normal(0, 0.2, n_samples)
        
        model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
        model.fit(x_tr[:, np.newaxis], y_tr)
        
        all_preds[d_idx, i, :] = model.predict(x_test[:, np.newaxis])

ตอนนี้เราจะคำนวณหาค่าความเอนเอียงกำลังสอง (bias$^2$) ความแปรปรวน (variance) และความคลาดเคลื่อนที่คาดหวังรวม (total expected error) ตลอดอินพุตทดสอบ โดยหาค่าเฉลี่ยรวมทุกจุดทดสอบครับ

In [ ]:
bias_sq_results = []
variance_results = []
total_error_results = []

for d_idx, degree in enumerate(degrees_sweep):
    preds_d = all_preds[d_idx, :, :]
    
    mean_pred = np.mean(preds_d, axis=0)
    
    # Bias^2
    bias_sq = np.mean((mean_pred - y_true) ** 2)
    bias_sq_results.append(bias_sq)
    
    # Variance
    variance = np.mean(np.var(preds_d, axis=0))
    variance_results.append(variance)
    
    # Expected squared error: Bias^2 + Variance + Irreducible Noise (0.2^2 = 0.04)
    total_error = bias_sq + variance + 0.04
    total_error_results.append(total_error)

# Plot the Bias-Variance U-Curve
plt.figure(figsize=(10, 6))
plt.plot(degrees_sweep, bias_sq_results, color='red', marker='o', label='Bias² (Underfitting indicator)')
plt.plot(degrees_sweep, variance_results, color='blue', marker='s', label='Variance (Overfitting indicator)')
plt.plot(degrees_sweep, total_error_results, color='black', marker='^', linewidth=2.5, label='Total Expected Error')
plt.xlabel('Polynomial Degree (Model Complexity)')
plt.ylabel('Error Value')
plt.title('Numerical Bias-Variance Tradeoff Curve')
plt.ylim(0, 0.5)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

ข้อสังเกต:
-   เมื่อระดับดีกรีพหุนามเพิ่มขึ้น **ค่าความเอนเอียงกำลังสอง (Bias$^2$) จะลดลง** อย่างรวดเร็ว
-   เมื่อระดับดีกรีเกินกว่า 4 **ค่าความแปรปรวน (Variance) จะพุ่งสูงขึ้น** อย่างรวดเร็ว
-   **ความคลาดเคลื่อนที่คาดหวังรวม (Total Expected Error)** จะมีลักษณะเป็นรูปตัว U โดยมีจุดต่ำสุดอยู่ที่ **Degree 4** (ซึ่งเป็นความซับซ้อนของโมเดลที่เหมาะสมที่สุดครับ)

## 💡 ความเชื่อมโยงกับคอมพิวเตอร์วิชันและ YOLO
*   **โมเดลขนาด Nano กับ Extra Large:** 
    -   `yolo11n` (Nano, พารามิเตอร์ 2.6M): มีความซับซ้อนต่ำ ความเอนเอียงสูง โมเดลทำงานเร็วแต่มีแนวโน้มเรียนรู้น้อยเกินไป (underfit) กับวัตถุขนาดเล็กหรือวัตถุที่ซับซ้อนในแผนภาพ PTT
    -   `yolo11x` (Extra Large, พารามิเตอร์ 56.9M): มีความซับซ้อนสูง ความเอนเอียงต่ำ แต่มีความแปรปรวนสูง (high variance) หากฝึกฝนด้วยชุดข้อมูลขนาดเล็ก (เช่น ภาพถ่าย 50 ภาพ) โมเดลจะเกิดการเรียนรู้มากเกินไป (overfit) โดยใช้วิธีท่องจำพิกเซลของพื้นหลังแทนครับ
*   **การต่อสู้กับความแปรปรวน (ลดการ Overfitting):** 
    -   รวบรวมข้อมูลฝึกฝนให้มากขึ้น (ซึ่งจะช่วยเลื่อนจุดต่ำสุดของเส้นโค้งรูปตัว U ไปทางขวา ทำให้เราสามารถใช้โมเดลที่มีความซับซ้อนสูงขึ้นได้)
    -   การกำหนดรูปแบบควบคุม (Regularization): ใช้ Weight Decay (L2 penalty) หรือ Dropout
    -   การเพิ่มข้อมูลเชิงสังเคราะห์ (Data Augmentation): สร้างภาพเฟรมฝึกฝนที่มีการพลิกรูป (flipped) ซูม (zoomed) หรือทำให้เบลอ (blurred)